# Week 1 quickstart

Runs the whole week-1 pipeline on the **real** data: download → clean → features → benchmarks → Poisson → Setup A models.

**Colab:** put your repo URL in the first cell, then *Runtime → Run all*. **Local:** skip the first cell and start Jupyter from the repo root.

In [ ]:
# Colab only: clone the repo and install packages
REPO_URL = "https://github.com/<your-username>/football-prediction.git"   # <-- change this
import os, sys
if "google.colab" in sys.modules and not os.path.exists("football-prediction"):
    !git clone $REPO_URL
    %cd football-prediction
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

## 1. Download and clean (Lyzing)
Read the sanity-check output. Every season should have 380 matches and 20 teams. Look through the team-name list for duplicates.

In [ ]:
from src.load_data import main as load_main, read_matches
load_main()
m = read_matches()
m.head()

## 2. Features (Darareach, Lyzing)
Pick 3 random matches and check a few values by hand against earlier rows (leakage check).

In [ ]:
from src.features import build_features, SETUPS
f = build_features(m)
f.to_csv("data/processed/features.csv", index=False)
print("missing in setup C:", int(f[SETUPS['C']].isna().sum().sum()))
f.sample(3, random_state=1)[["date", "home", "away", "result", "elo_diff", "home_form_pts5", "away_form_pts5", "home_rest_days", "h2h_score", "h2h_n"]]

## 3. Benchmarks (Cheypol)
Copy these rows into the Results Log. Benchmarks can be scored on test any time, because nothing is tuned.

In [ ]:
from src.evaluate import benchmark_table, split, evaluate
bench = benchmark_table(f)
bench

## 4. Walk-forward Poisson (Chhoungkorng)
Takes about a minute. These probabilities are the Poisson baseline and the extra Setup D features.

In [ ]:
from src.poisson import walk_forward_probs, fill_missing_probs
probs = walk_forward_probs(m)
probs.to_csv("data/processed/poisson_probs.csv", index=False)
print("unseen-team matches filled with frequencies:", int(probs.pois_missing.sum()))
f = f.merge(fill_missing_probs(probs), on="match_id", how="left")
_, val, _ = split(f)
evaluate(val.label, val[["pois_pH", "pois_pD", "pois_pA"]].values)

## 5. Setup A on validation (Cheypol)
Default settings only. Tuning comes in week 3. **Do not touch the test split here.**

In [ ]:
from src.models import run_validation
run_validation(f, {"A": SETUPS["A"]})

## Next (week 2)
- Look at the setup A results above next to the benchmarks. Above ~60% accuracy means check for leakage.
- `run_validation(f, SETUPS)` runs all four setups once features are checked.
- Run `python -m pytest -q` after any change to `src/features.py`.